# 03 — EMLM: کمیته‌ی وزن‌دار مدل‌ها (جداول ۹ و ۱۰ مقاله) — دیتاست-مستقل

**نحوه‌ی کار:** از نتایج نوت‌بوک ۲ (`results/datasetX_table5..8.csv`) بهترین سلولِ هر آزمایش انتخاب و کمیته ساخته می‌شود؛ وزن هر عضو از AUC درونی tuning گرفته می‌شود (کف 0.5 = شانس تصادفی، تا جلوگیری از وزن منفی/بی‌اثر)؛ ارزیابی روی فولدهای بیرونیِ **مشترک** تا ترکیب احتمالات معنادار باشد.

**`FAST=True`**: smoke run. **`FAST=False`**: paper protocol — nested CV 5×4×1 with full/Bayesian grids.

In [ ]:
# Settings — change these two lines only
DATASET = "dataset1"     # "dataset1" | "dataset2"
FAST = False

import sys, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from ensemble import build_committee, evaluate_committee
from preprocessing import dedup, load_standard

df, FEATS = load_standard(f"data/{DATASET}_standard.csv")
df = dedup(df)
X, y = df[FEATS], df["CLASS_code"]
print(f"{DATASET}: {df.shape[0]} ردیف × {len(FEATS)} فیچر | کلاس‌ها: {y.value_counts().sort_index().to_dict()}")

In [ ]:
# 1) ساخت کمیته از بهترین سلول هر آزمایش (جداول ۵-۸)
committee = build_committee("results", DATASET)
print("اعضای کمیته (بهترین سلول هر آزمایش از نوت‌بوک ۲):")
print(committee.to_string(index=False))
committee.to_csv(f"results/{DATASET}_committee.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 2) ارزیابی اعضا + EMLM روی فولدهای بیرونی مشترک
mem_df, emlm_summary, emlm_df, oof = evaluate_committee(X, y, committee, fast=FAST)
print("=== جدول اعضا (میانگین روی فولدها) ===")
print(mem_df.to_string(index=False))
print("\n=== EMLM (جدول ۹ مقاله) ===")
print(f"AAC={emlm_summary['aac']}±{emlm_summary['aac_std']} | ACC={emlm_summary['acc']} | AUC={emlm_summary['auc']} | فولدها={emlm_summary['n_folds']}")
mem_df.to_csv(f"results/{DATASET}_emlm_members.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([emlm_summary]).to_csv(f"results/{DATASET}_table9.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 3) گزارش کلاس‌محور (جدول ۱۰ مقاله) — عملکرد EMLM روی هر کلاس
from sklearn.metrics import classification_report, confusion_matrix
print("Confusion Matrix (OOF، ردیف=واقعی):")
print(confusion_matrix(oof["y_true"], oof["y_pred"], labels=sorted(oof["y_true"].unique())))
print("\nPer-class report:")
print(classification_report(oof["y_true"], oof["y_pred"], digits=4))
oof.to_csv(f"results/{DATASET}_emlm_oof.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 4) جمع‌بندی: EMLM در برابر بهترین تک‌مدل
best_single = mem_df.loc[mem_df["aac"].idxmax()]
print(f"بهترین تک‌عضو: {best_single['model']} ({best_single['selector']}, prep={best_single['prep']}) → AAC={best_single['aac']}")
print(f"EMLM → AAC={emlm_summary['aac']} (Δ={round(emlm_summary['aac'] - best_single['aac'], 4)})")
print("\n⚠️ نکته‌ی مقایسه با مقاله: مقاله ACC=0.998 روی دیتاست پس از حذف فقط ۷ تکراری گزارش کرده؛ "
      "ما dedup سخت‌گیرانه‌تر (۱۷۴ ردیف در dataset2) و معیار متعادل AAC را هم گزارش می‌کنیم — تفاوت انتظارشده است.")
emlm_df.to_csv(f"results/{DATASET}_emlm_folds.csv", index=False, encoding="utf-8-sig")